# Module 1 — Classical Computer Vision
**Assignments 1.1 (simple filters) and 1.2 (boundary detection).**

Before neural networks, people designed every image operation by hand. In this notebook you apply those operations to
**your own photos**: you see what each one measures and where it fails. Those failures are why the rest of the course
uses convolutional networks.

**How to use this notebook:** run the cells from top to bottom (*Run All*). Under every result, write one sentence on
what the operation measured, in your own words, based on what you see.

## 0. Setup — works on your computer and on Google Colab
- **On your computer:** open this notebook from your project folder, choose the kernel *Python (mai630)*, and make sure your photos are in `data/m1_images/` and your `src/classical.py` passes its tests.
- **On Colab:** run the next cell. It asks you to upload a .zip of your project folder and then your photos.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import files
    found = list(Path("/content").glob("*/src/classical.py"))
    if not found:
        print("Upload a .zip of your project folder (with src/classical.py inside):")
        for name, content in files.upload().items():
            Path("/content", name).write_bytes(content)
            !unzip -q -o "/content/{name}" -d /content
        found = list(Path("/content").glob("*/src/classical.py"))
    os.chdir(found[0].parents[1])
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
IMAGE_DIR = ROOT / "data" / "m1_images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and len([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]) < 3:
    print("Upload at least three photos of your own:")
    for name, content in files.upload().items():
        (IMAGE_DIR / name).write_bytes(content)

NAMES = sorted(p.name for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
assert len(NAMES) >= 3, f"Put at least 3 photos in {IMAGE_DIR} (found {len(NAMES)})"
print("Folder:", ROOT)
print("Images:", NAMES)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src import classical as cl


def show_row(images, titles, size=3.6):
    """Draw images side by side: 2-D arrays in grey (0 = black, 255 = white), 3-D arrays in colour."""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.75))
    for ax, img, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255) if img.ndim == 2 else ax.imshow(img)
        ax.set_title(title, fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()

## Part A — Assignment 1.1: simple filters
### A0. What a digital image is
**What it is.** To a computer, a photo is a table of numbers (a *matrix*). Each cell is a **pixel**. In a greyscale image
each pixel is one number from 0 (black) to 255 (white). A colour image stores three numbers per pixel: red, green
and blue.

**Watch out:** OpenCV reads the channels in the order **B, G, R**. `cl.load_rgb` converts them to R, G, B once, so the
colours look right when we draw them.

In [ ]:
rgb = {n: cl.load_rgb(IMAGE_DIR / n) for n in NAMES}
for n, img in rgb.items():
    print(f"{n}: shape={img.shape} (height, width, channels)  dtype={img.dtype}  pixel(50,50) RGB={img[50, 50]}")

**How to read it:** `shape` is the size of the table. `uint8` means whole numbers from 0 to 255. The three numbers of
a pixel are its amount of red, green and blue.

### A1. Greyscale conversion
**Why:** many classical operations need one number per pixel, not three.
**How:** a *weighted* average, `grey = 0.299 R + 0.587 G + 0.114 B`. Green weighs most because the human eye is most
sensitive to it.

In [ ]:
grey = {n: cl.to_grey(rgb[n]) for n in NAMES}
for n in NAMES:
    show_row([rgb[n], grey[n]], [f"{n} original", f"greyscale {grey[n].shape}"])

**What it measured (write after running the cell above):** …

### A2. Brightness and contrast
**How:** `new = α · pixel + β`. β (beta) adds the same amount to every pixel, which changes **brightness**. α (alpha)
stretches the distance between dark and light, which changes **contrast**. The result is clipped to 0–255.
**Watch out:** whatever is pushed past 255 becomes pure white, and that detail is lost for good.

In [ ]:
for n in NAMES:
    show_row([grey[n], cl.brightness_contrast(grey[n], 1.0, 60), cl.brightness_contrast(grey[n], 1.8, -60)],
             [f"{n} grey", "brightness β = +60", "contrast α = 1.8, β = −60"])

**What it measured (write after running the cell above):** …

### A3. Blurring at two levels
**What it is:** each pixel is replaced by an average of its neighbours. The neighbourhood is a small square window,
the **kernel**. In a Gaussian blur the closest neighbours weigh more.
**Why:** to reduce noise, the small random errors every camera adds.
**The cost:** a larger kernel averages a wider area, so it also erases detail and edges.

In [ ]:
blur = {n: {k: cl.gaussian_blur(grey[n], k) for k in (5, 21)} for n in NAMES}
for n in NAMES:
    show_row([grey[n], blur[n][5], blur[n][21]], [f"{n} grey", "Gaussian 5 × 5", "Gaussian 21 × 21"])

**What it measured (write after running the cell above):** …

### A4. Thresholding
**What it is:** choose a number T. Pixels brighter than T become white and the rest black.
**Otsu's method** chooses T for you: it tries every value and keeps the one that best separates the histogram into two
groups. We blur first (5 × 5) so the noise does not drive the choice.
**How to read it:** thresholding separates *bright from dark*. It does not know what an object is.

In [ ]:
for n in NAMES:
    T, binary = cl.threshold_otsu(blur[n][5])
    print(f"{n}: Otsu chose T = {T}")
    show_row([grey[n], binary], [f"{n} grey", f"Otsu threshold, T = {T}"])

**What it measured (write after running the cell above):** …

### A5. A convolution with a kernel you write yourself
**What it is:** a **convolution** slides the kernel over the image. At each position it multiplies every pixel of the
window by the weight in the same place and adds the results. That sum is the new pixel.

Our kernel has 5 in the centre, −1 above, below, left and right, and 0 in the corners. The weights add up to 1, so flat
areas keep their value.

**Write your prediction before running:** where a pixel differs from its neighbours (a crack, the rim of a hole) that
difference is amplified. So edges and texture should look sharper and grainier, and flat areas should not change.

In [ ]:
kernel = cl.KERNELS["sharpen"]
print(kernel, "\nsum of weights =", kernel.sum())
for n in NAMES:
    show_row([grey[n], cl.apply_kernel(grey[n], kernel)], [f"{n} grey", "my 3 × 3 kernel"])

# Check: the same operation written out by hand gives the same numbers as OpenCV
import cv2
check = cl.convolve_by_hand(grey[NAMES[0]][:80, :80], kernel)
ref = cv2.filter2D(grey[NAMES[0]][:80, :80].astype(np.float32), -1, kernel, borderType=cv2.BORDER_REFLECT_101)
print("largest difference, by hand vs OpenCV:", float(np.abs(check - ref).max()))

**What it measured (write after running the cell above):** …

## Part B — Assignment 1.2: boundary detection
**What an edge is:** a place where the brightness **changes** quickly, not a colour. To find change we compare
neighbouring pixels. Mathematically that is a **derivative**. All three operators use the 5 × 5-blurred greyscale, so
they see the same input.

### B1. Sobel — the first derivative
Two 3 × 3 kernels estimate the change from left to right (Gx) and from top to bottom (Gy). The magnitude
√(Gx² + Gy²) says how strong the change is, in any direction.

In [ ]:
sobel_mag = {}
for n in NAMES:
    gx, gy, sobel_mag[n] = cl.sobel(blur[n][5])
    show_row([rgb[n], cv2.convertScaleAbs(gx), cv2.convertScaleAbs(gy), sobel_mag[n]],
             [n, "Gx (vertical edges)", "Gy (horizontal edges)", "magnitude"])

**What it measured (write after running the cell above):** …

### B2. Laplacian — the second derivative
It measures where the *change itself* changes, in every direction at once. It reacts very strongly to fine texture
and noise. That is why we blur first.

In [ ]:
lap = {n: cl.laplacian(blur[n][5]) for n in NAMES}
for n in NAMES:
    show_row([rgb[n], lap[n]], [n, "Laplacian"])

**What it measured (write after running the cell above):** …

### B3. Canny at three pairs of thresholds
Canny works in four steps:
1. blur;
2. compute the gradient (as Sobel does);
3. keep only the thin ridge of each edge;
4. apply **two thresholds**: pixels above `high` are edges, pixels below `low` are discarded, and pixels in between
   are kept only if they connect to a strong edge.

With two thresholds, a weak stretch of an edge survives when it continues a strong one.

In [ ]:
PAIRS = [(30, 90), (100, 200), (200, 400)]
canny = {n: {p: cl.canny(blur[n][5], *p) for p in PAIRS} for n in NAMES}
for n in NAMES:
    print(n, "edge pixels:", {p: cl.edge_pixels(canny[n][p]) for p in PAIRS})
    show_row([rgb[n]] + [canny[n][p] for p in PAIRS], [n] + [f"Canny {p}" for p in PAIRS])

**What it measured (write after running the cell above):** …

### B4. The three operators side by side

In [ ]:
for n in NAMES:
    show_row([rgb[n], sobel_mag[n], lap[n], canny[n][(100, 200)]], [n, "Sobel", "Laplacian", "Canny (100, 200)"])

### B5. One image where the operators disagree — and why

**What it measured (write after running the cell above):** …

### B6. One image or region where every operator fails — and what defeats them

**What it measured (write after running the cell above):** …

## Short written comparison

**What it measured (write after running the cell above):** …